In [3]:
import os
from dotenv import load_dotenv
from langchain_openai.chat_models import ChatOpenAI
from langchain_openai.embeddings import OpenAIEmbeddings
from langchain_core.output_parsers import StrOutputParser
import fitz  # PyMuPDF
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.vectorstores import FAISS
from langchain.prompts import PromptTemplate
from operator import itemgetter


/Users/Shared/RAG-for-drug-pdf-files/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [4]:
# Load environment variables from .env file
load_dotenv()

# Retrieve API key from .env file
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
MODEL = os.getenv("MODEL")
PDF_FOLDER = os.getenv("PDF_FOLDER")
ONEDRIVE_PATH_PDF = os.getenv("ONEDRIVE_PATH_PDF")

# Initialize GPT model and embeddings from OpenAI
model = ChatOpenAI(openai_api_key=OPENAI_API_KEY, model=MODEL, temperature=0)
embeddings = OpenAIEmbeddings(openai_api_key=OPENAI_API_KEY)


In [3]:
# Example GPT model invocation
response = model.invoke("Tell me a joke")
print(response)

content='Why was the math book sad?\n\nBecause it had too many problems.' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 11, 'total_tokens': 26, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'finish_reason': 'stop', 'logprobs': None} id='run-721a543d-e849-4507-b665-3acffeb778a9-0' usage_metadata={'input_tokens': 11, 'output_tokens': 15, 'total_tokens': 26, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}


In [4]:
response

AIMessage(content='Why was the math book sad?\n\nBecause it had too many problems.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 11, 'total_tokens': 26, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'finish_reason': 'stop', 'logprobs': None}, id='run-721a543d-e849-4507-b665-3acffeb778a9-0', usage_metadata={'input_tokens': 11, 'output_tokens': 15, 'total_tokens': 26, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [5]:
parser = StrOutputParser()

chain = model | parser 
chain.invoke("Tell me a joke")

'Why did the scarecrow win an award? Because he was outstanding in his field!'

In [6]:
# Check if the PDF folder is defined
if not PDF_FOLDER:
    raise ValueError("The path to the PDF folder is not defined in the .env file.")

In [8]:

# Variable to store text from all PDF files
all_text = ""

# Iterate over all PDF files in the folder
for filename in os.listdir(PDF_FOLDER):
    if filename.endswith(".pdf"):
        pdf_path = os.path.join(PDF_FOLDER, filename)
        print(f"Processing file")

        try:
            # Open the PDF using PyMuPDF (fitz)
            doc = fitz.open(pdf_path)

            # Iterate over pages and collect text
            for page_num in range(doc.page_count):
                page = doc.load_page(page_num)
                all_text += page.get_text()  # Extract text from each page

        except Exception as e:
            print(f"Error processing file {filename}: {e}")

# After collecting all text, split it into smaller chunks
splitter = RecursiveCharacterTextSplitter(chunk_size=1500, chunk_overlap=100)
chunks = splitter.split_text(all_text)

# Print the number of chunks
print(f"Number of chunks: {len(chunks)}")

# Display the first chunk
if chunks:
    print(f"Content of the first chunk:\n{chunks[0]}")


Processing file: Charakterystyka-47940-2023-09-19-15080_D-2023-10-10.pdf
Processing file: Charakterystyka-9247-2023-05-04-13593_A-2023-05-11.pdf
Processing file: Charakterystyka-35622-20210114000000-2450_M-20210115000445.pdf
Processing file: Charakterystyka-11993-20180509000000-4285_A-20180620000952.pdf
Processing file: Charakterystyka-44749-2022-08-10-1926_D-2022-09-20.pdf
Processing file: Charakterystyka-45810-2023-05-15-1858_D-2023-06-13.pdf
Processing file: Charakterystyka-31773-2023-12-20-15983_M-2023-12-21.pdf
Processing file: Charakterystyka-23921-2022-03-28-11693_B-2022-07-15.pdf
Processing file: Charakterystyka-36499-2023-08-10-14066_M-2023-08-19.pdf
Processing file: Charakterystyka-31740-2022-11-28-9423_N-2022-12-12.pdf
Processing file: Charakterystyka-40773-2024-03-19-17459_B-2024-04-10.pdf
Processing file: Charakterystyka-19175-2023-08-24-14945_A-2023-09-01.pdf
Processing file: Charakterystyka-36006-2021-12-02-10922_B-2022-02-05.pdf
Processing file: Charakterystyka-30387-20

In [11]:
# Load or create embeddings
def load_or_create_embeddings(source_text_chunks, onedrive_path):
    faiss_index_path = os.path.join(onedrive_path, "index.faiss")

    if os.path.exists(faiss_index_path):
        print(f"Loading existing embeddings from: {faiss_index_path}")
        vectorstore = FAISS.load_local(onedrive_path, embeddings, allow_dangerous_deserialization=True)
    else:
        print(f"Embeddings do not exist. Generating new embeddings and saving to: {faiss_index_path}")
        vectorstore = FAISS.from_texts(source_text_chunks, embeddings)
        vectorstore.save_local(onedrive_path)

    return vectorstore

In [12]:
pdf_vectorstore = load_or_create_embeddings(chunks, ONEDRIVE_PATH_PDF)

Loading existing embeddings from: /Users/Shared/RAG-for-drug-pdf-files/database/index.faiss


Key Differences:
Direct Search (similarity_search) vs. Search Using Retriever (retriever.invoke())

similarity_search is a more direct method where the search is solely based on the similarity of text embeddings. You can control the number of results using the k parameter.
retriever.invoke() can apply more advanced query and result processing mechanisms (e.g., filtering, additional optimization steps). This may result in different, even subtly varying, outcomes.
Control Over the Number of Results:
In similarity_search, you can explicitly control the number of results returned by adjusting the k parameter.
In retriever.invoke(), the number of results depends on the retriever's implementation. You don’t have direct control over the number of results unless configured explicitly.
When to Use?
similarity_search:
Use it when you want a straightforward and quick way to find the most similar documents based on embeddings. It’s especially useful when you need full control over the number of results.

retriever.invoke():
Use it when you need a more flexible and advanced search mechanism. It’s suitable if you want to combine the results with other functionalities, such as applying additional filtering or processing that can influence the search or result handling.

In [13]:
# Test the functionality with a query
query = "What are the side effects of the drug Ifapidin?"

# Perform similarity search on the query
docs = pdf_vectorstore.similarity_search(query, k=3)

# Display the results
for i, doc in enumerate(docs):
    print(f"Result {i+1}: {doc.page_content}")

Result 1: krwiotwórczego.  
 
Decyzję o wznowieniu leczenia produktem Ifapidin należy podejmować na podstawie oceny objawów 
klinicznych i wyników badań laboratoryjnych.   
 
Reakcje krzyżowe między tienopirydynami  
Należy zebrać od pacjentów wywiad dotyczący nadwrażliwości na inną tienopirydynę (na przykład 
klopidogrel, prasugrel), ponieważ opisywano krzyżowe reakcje alergiczne między tienopirydynami 
(patrz punkt 4.8). Tienopirydyny mogą powodować łagodne do ciężkich reakcje alergiczne, takie jak 
wysypka, obrzęk naczynioruchowy lub hematologiczne reakcje krzyżowe, jak trombocytopenia i 
neutropenia. Pacjenci, u których wcześniej występowała reakcja alergiczna i (lub) hematologiczna na 
jakąś tienopirydynę, mogą być zagrożeni większym ryzykiem wystąpienia takiej samej lub innej 
 
 
 
 
4
reakcji na inny lek z grupy tienopirydyn. U pacjentów ze stwierdzoną alergią na tienopirydyny zaleca 
się obserwację w kierunku objawów nadwrażliwości.   
 
Hemostaza 
Produkt należy stosować ze s

In [14]:
# Test the functionality with a query
query = "What is the main chemical substance in Bisocard?"

# Perform similarity search on the query
docs = pdf_vectorstore.similarity_search(query, k=3)

# Display the results
for i, doc in enumerate(docs):
    print(f"Result {i+1}: {doc.page_content}")

Result 1: Data wydania pierwszego pozwolenia na dopuszczenie do obrotu: 29 kwietnia 2004 
Data ostatniego przedłużenia pozwolenia: 23 września 2013 
 
 
10. 
DATA ZATWIERDZENIA LUB CZĘŚCIOWEJ ZMIANY CHARAKTERYSTYKI 
PRODUKTU LECZNICZEGO 
 
27.07.2021 
 
1 
 
CHARAKTERYSTYKA PRODUKTU LECZNICZEGO 
 
 
1.  
NAZWA PRODUKTU LECZNICZEGO 
 
Bisocard, 5 mg, tabletki powlekane 
Bisocard, 10 mg, tabletki powlekane 
 
 
2.  
SKŁAD JAKOŚCIOWY I ILOŚCIOWY  
 
Bisocard, 5 mg, tabletki powlekane:  
Jedna tabletka powlekana zawiera 5 mg bisoprololu fumaranu (Bisoprololi fumaras). 
Substancja pomocnicza o znanym działaniu: laktoza jednowodna w ilości 120 mg. 
Bisocard, 10 mg, tabletki powlekane:  
Jedna tabletka powlekana zawiera 10 mg bisoprololu fumaranu (Bisoprololi fumaras).  
Substancja pomocnicza o znanym działaniu: laktoza jednowodna w ilości 115 mg. 
 
Pełny wykaz substancji pomocniczych, patrz punkt 6.1. 
 
 
3.  
POSTAĆ FARMACEUTYCZNA 
 
Tabletka powlekana. 
 
5 mg: Jasnożółte, okrągłe, obust

In [15]:
retriever = pdf_vectorstore.as_retriever()
retriever.invoke("What are the side effects of the drug Ifapidin?")
     

[Document(id='204416e2-6e77-4445-8e1c-33f9f347ab05', metadata={}, page_content='krwiotwórczego.  \n \nDecyzję o wznowieniu leczenia produktem Ifapidin należy podejmować na podstawie oceny objawów \nklinicznych i wyników badań laboratoryjnych.   \n \nReakcje krzyżowe między tienopirydynami  \nNależy zebrać od pacjentów wywiad dotyczący nadwrażliwości na inną tienopirydynę (na przykład \nklopidogrel, prasugrel), ponieważ opisywano krzyżowe reakcje alergiczne między tienopirydynami \n(patrz punkt 4.8). Tienopirydyny mogą powodować łagodne do ciężkich reakcje alergiczne, takie jak \nwysypka, obrzęk naczynioruchowy lub hematologiczne reakcje krzyżowe, jak trombocytopenia i \nneutropenia. Pacjenci, u których wcześniej występowała reakcja alergiczna i (lub) hematologiczna na \njakąś tienopirydynę, mogą być zagrożeni większym ryzykiem wystąpienia takiej samej lub innej \n \n \n \n \n4\nreakcji na inny lek z grupy tienopirydyn. U pacjentów ze stwierdzoną alergią na tienopirydyny zaleca \nsię ob

In [16]:
retriever.invoke("What is the main chemical substance in Bisocard?")

[Document(id='669ecbde-8201-4d4c-b9f1-92373cf77147', metadata={}, page_content='Data wydania pierwszego pozwolenia na dopuszczenie do obrotu: 29 kwietnia 2004 \nData ostatniego przedłużenia pozwolenia: 23 września 2013 \n \n \n10. \nDATA ZATWIERDZENIA LUB CZĘŚCIOWEJ ZMIANY CHARAKTERYSTYKI \nPRODUKTU LECZNICZEGO \n \n27.07.2021 \n \n1 \n \nCHARAKTERYSTYKA PRODUKTU LECZNICZEGO \n \n \n1.  \nNAZWA PRODUKTU LECZNICZEGO \n \nBisocard, 5 mg, tabletki powlekane \nBisocard, 10 mg, tabletki powlekane \n \n \n2.  \nSKŁAD JAKOŚCIOWY I ILOŚCIOWY  \n \nBisocard, 5 mg, tabletki powlekane:  \nJedna tabletka powlekana zawiera 5 mg bisoprololu fumaranu (Bisoprololi fumaras). \nSubstancja pomocnicza o znanym działaniu: laktoza jednowodna w ilości 120 mg. \nBisocard, 10 mg, tabletki powlekane:  \nJedna tabletka powlekana zawiera 10 mg bisoprololu fumaranu (Bisoprololi fumaras).  \nSubstancja pomocnicza o znanym działaniu: laktoza jednowodna w ilości 115 mg. \n \nPełny wykaz substancji pomocniczych, patrz

In [17]:
# Initialize GPT model for direct questions
response = model.invoke("Who is the president of Poland?")

# Display GPT response
print(response)

content='The current president of Poland is Andrzej Duda.' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 12, 'prompt_tokens': 14, 'total_tokens': 26, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'finish_reason': 'stop', 'logprobs': None} id='run-3147562a-7f20-44d7-8891-1655a3067d65-0' usage_metadata={'input_tokens': 14, 'output_tokens': 12, 'total_tokens': 26, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}


In [18]:
parser = StrOutputParser()

chain = model | parser 
print(chain.invoke("Who is the president of the United States?"))

As of September 2021, the President of the United States is Joe Biden.


In [20]:
template = """
You are an assistant specializing in medical and pharmaceutical information, providing precise answers based on
the provided context. Your task is to retrieve information from the official drug leaflets (ChPLs).

- Answer the question in English based on the given context.
- Include only the relevant details from the context in your answer.
- If the context does not contain enough information to answer the question, respond with: "I don't know based on the provided information."
- Avoid including unrelated information.

Context: {context}

Question: {question}
"""
prompt = PromptTemplate.from_template(template)
print(prompt.format(context="Here is some context", question="Here is a question"))


You are an assistant specializing in medical and pharmaceutical information, providing precise answers based on
the provided context. Your task is to retrieve information from the official drug leaflets (ChPLs).

- Answer the question in English based on the given context.
- Include only the relevant details from the context in your answer.
- If the context does not contain enough information to answer the question, respond with: "I don't know based on the provided information."
- Avoid including unrelated information.

Context: Here is some context

Question: Here is a question



In [21]:
chain = prompt | model | parser

chain.invoke({
    "context": "Anna's sister is Susan", 
    "question": "Who is Susan's sister?"
})


'Anna'

In [22]:
chain = (
    {
        "context": itemgetter("question") | retriever,
        "question": itemgetter("question"),
    }
    | prompt
    | model
    | parser
)

In [23]:
# Define questions to test the chain
questions = [
    "Give me the names of chemical substances that interact with Atoris",
    "Does the drug IPP have the same indications as Omeprazole?",
    "Does Bisocard lower blood pressure?",
    "Did you base your answers to the above questions only on the context?",
    "What is the composition of the drug Zomiren?",
    "Does Ifapidin interact with Bisocard?",
    "Do Bibloc and Bisocard have the same chemical composition?"
]

# Loop through questions and test the chain
for question in questions:
    print(f"Question: {question}")
    print(f"Answer: {chain.invoke({'question': question})}")
    print("*************************\n")

Question: Give me the names of chemical substances that interact with Atoris
Answer: I don't know based on the provided information.
*************************

Question: Does the drug IPP have the same indications as Omeprazole?
Answer: I don't know based on the provided information.
*************************

Question: Does Bisocard lower blood pressure?
Answer: Based on the provided information, yes, Bisocard (bisoprolol) can lower blood pressure. The simultaneous use of bisoprolol with certain medications can lead to further lowering of blood pressure. Additionally, bisoprolol has been shown to reduce the total mortality rate and the number of heart failure episodes requiring hospitalization, indicating its effectiveness in managing blood pressure and heart-related conditions.
*************************

Question: Did you base your answers to the above questions only on the context?
Answer: Yes, I based my answers solely on the provided context.
*************************

Question: W

In [24]:
# Define questions to test the chain
questions = [
    "Give me the names of drugs used for the flu",
    "Does the drug Kalipoz have the same indications as IPP?",
    "Does Acard lower blood pressure?",
    "What is the best drug for strong pain?",
    "Is Zinnat used for stomach pain?",
    "Find me similar drugs to Ifapidin",
    "What is the active substance in Ibuprom?"
]

# Loop through questions and test the chain
for question in questions:
    print(f"Question: {question}")
    print(f"Answer: {chain.invoke({'question': question})}")
    print("*************************\n")

Question: Give me the names of drugs used for the flu
Answer: I don't know based on the provided information.
*************************

Question: Does the drug Kalipoz have the same indications as IPP?
Answer: I don't know based on the provided information.
*************************

Question: Does Acard lower blood pressure?
Answer: I don't know based on the provided information.
*************************

Question: What is the best drug for strong pain?
Answer: Based on the provided context, the best drug for strong pain would be APAP intense, which is intended for adults aged 18 and older. It is suitable for treating pain requiring stronger analgesic action than ibuprofen or paracetamol used separately. The recommended dosage for adults is one tablet up to 3 times a day, with a minimum of 6 hours between doses, not exceeding 6 tablets (1200 mg ibuprofen, 3000 mg paracetamol) in 24 hours.
*************************

Question: Is Zinnat used for stomach pain?
Answer: I don't know base